In [ ]:
Цели занятия
Концепция Event Loop:

Познакомиться с основами работы цикла событий (event loop), который управляет асинхронными задачами.
Понять, как цикл событий обрабатывает события и управляет асинхронными вызовами.
Старый и новый синтаксис асинхронных вызовов:

Рассмотреть различия между синтаксисом async/await, который был введён в Python 3.5,
и более ранними подходами к асинхронному программированию, такими как использование коллбэков и библиотеки, такие как twisted.
Высоко- и низкоуровневое API asyncio:

Изучить высокоуровневые API, такие как asyncio.run(), которые упрощают запуск асинхронных функций.
Рассмотреть низкоуровневые API, такие как loop.create_task() и loop.run_until_complete(), 
    которые дают больше контроля над работой цикла событий.
Внутреннее устройство awaitable объектов:

Понять, что такое awaitable объекты (функции и корутины) и как они работают внутри asyncio.
Изучить, как реализованы корутины и какие внутренние механизмы задействованы при их использовании.


In [ ]:
loop = asyncio.get_event_loop()
task = loop.create_task(main())
loop.run_until_complete(task)
future = asyncio.Future()

loop.close()


In [2]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

async def say_hello():
    print("Hello")
    await asyncio.sleep(1)
    print("Hello2")

async def main():
    await asyncio.gather(say_hello(), say_hello())

asyncio.run(main())


Hello
Hello
Hello2
Hello2


In [ ]:
В asyncio два ключевых понятия — это Task и Future. Они используются для управления асинхронными операциями и 
имеют разные уровни абстракции и применения. 

Task vs. Future
Future
Что это?
Future представляет собой объект, который будет содержать результат асинхронной операции, но пока результат не доступен. 
Он позволяет проверить, завершена ли операция, и получить её результат, когда она завершится.

Как это работает?
Вы можете создать Future вручную, но чаще всего это делается в рамках задач, которые управляются asyncio.

Пример:

In [ ]:
import asyncio
import nest_asyncio

nest_asyncio.apply()
async def fetch_data():
    await asyncio.sleep(2)
    return "some data"

async def main():
    loop = asyncio.get_event_loop()
    future = loop.create_future()

    asyncio.ensure_future(fetch_data())
    result = await future

    print(result)

asyncio.run(main())
    


In [3]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

async def worker(fut):
    await asyncio.sleep(1)
    fut.set_result("sdgsdg")

async def main():
    loop = asyncio.get_running_loop()
    future = loop.create_future()
    task = asyncio.create_task(worker(future))
    result = await future
    print(result)

asyncio.run(main())

    

sdgsdg


In [ ]:
Task
Что это?
Task — это специальный тип Future, который представляет собой корутину, которую нужно выполнить. Task управляет 
выполнением корутины и предоставляет дополнительные методы для работы с её статусом.

Как это работает?
Когда вы создаете Task, она автоматически добавляется в цикл событий и выполняется асинхронно.

Пример:

In [ ]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

async def fetch_data():
    await syncio.sleep(2)
    return "some data"

async def main():
    task = asyncio.create_task(fetch_data())

    print("waiting...")
    result = await task
    print(result)

asyncio.run(main())


In [ ]:
Разные уровни API фреймворка asyncio
Высокоуровневое API
asyncio.run(): Простая функция для запуска корутины, которая автоматически создает цикл событий 
и управляет его жизненным циклом.

asyncio.create_task(): Позволяет создать и запустить Task, которая будет выполняться асинхронно.

asyncio.gather(): Позволяет запускать несколько корутин параллельно и собирать результаты.

In [4]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

async def fetch_data(index):
    await asyncio.sleep(2)
    return f"data from task {index}"

async def bounded_fetch(semaphore, index):
    async with semaphore:
        result = await fetch_data(index)
        print(result)

async def main():
    num_tasks = 10
    max_concurrent_tasks = 3
    semaphore = asyncio.Semaphore(max_concurrent_tasks)

    tasks = [bounded_fetch(semaphore, i) for i in range(num_tasks)]
    await asyncio.gather(*tasks)

asyncio.run(main())



<3.11
async def main():
    try:
        result = await asyncio.wait_for(func(), timeout=2)
    except asyncio.TimeoutError:
        pass

>=3.11

async def main():
    try:
        async with asyncio.timeout(2):
            result = await func()
    except asyncio.TimeoutError:
        pass


# queue

async def producer(queue):
    for i in range(5):
        await queue.put(i)
        await asyncio.sleep(1)

async def consumer(queue):
    while True:
        item = await queue.get()
        queue.task_done()

async def main():
    queue = asyncio.Queue()
    await asyncio.gather(producer(queue), consumer(queue))
    
        






data from task 0
data from task 1
data from task 2
data from task 3
data from task 4
data from task 5
data from task 6
data from task 7
data from task 8
data from task 9


In [ ]:
Низкоуровневое API
loop.run_until_complete(): Позволяет запустить цикл событий до завершения заданной корутины.

loop.create_future(): Позволяет создавать Future вручную, чтобы управлять результатом асинхронной операции.

loop.run_forever(): Запускает цикл событий, который будет работать бесконечно, пока его не остановят вручную.

Пример с использованием низкоуровневого API

In [ ]:
import asyncio

async def fetch_data():
    await asyncio.sleep(2)
    return "result"

def main():
    loop = asyncio.get_event_loop()
    result = loop.run_until_complete(fetch_data())
    print(result)

if __name__ == "__main__":
    main()


In [ ]:
import asyncio

async def waiter(event):
    print("Waiting for event...")
    await event.wait()  # Ждет установки события
    print("Event is set!")

async def main():
    event = asyncio.Event()
    asyncio.create_task(waiter(event))
    await asyncio.sleep(2)
    event.set()  # Устанавливаем событие
    await asyncio.sleep(1)

asyncio.run(main())


In [ ]:
1. Event Loop
Цикл событий (event loop) — это основной компонент, который управляет выполнением асинхронных задач в Python. 
Он позволяет эффективно обрабатывать асинхронные операции, такие как I/O, таймеры и другие события, не блокируя основной поток выполнения.

Основные функции Event Loop:
Обработка событий: Цикл событий управляет очередью событий и вызывает соответствующие обработчики.
Запуск корутин: Он управляет выполнением корутин, создавая задачи (Tasks).
Синхронизация: Обеспечивает взаимодействие между разными асинхронными задачами.
Пример использования Event Loop

In [ ]:
server = loop.run_untill_complete(
    asyncio.start_server(handle_client, "localhost", 8080)
)

loop.run_forever()

In [ ]:
2. Async/Await
async и await — это ключевые слова, которые используются для определения асинхронных 
функций и указания на места, где выполнение может быть приостановлено.

async: Используется для объявления асинхронной функции (корутины). Такие функции возвращают coroutine object.

await: Используется для приостановки выполнения корутины до завершения другой асинхронной 
операции или awaitable объекта. Он может использоваться только внутри асинхронных функций.

Пример использования async/await

In [ ]:
import asyncio

async def fetch_data():
    print("sdf")
    await asyncio.sleep(2)
    print("dsg")
    return "sdf"

async def main():
    result = await fetch_data()
    print("sdfg")

asyncio.run(main())


In [ ]:
3. Awaitable Объекты
Awaitable — это любой объект, который может быть использован с await. 
Существует несколько типов awaitable объектов в Python:

Корутины: Функции, объявленные с использованием async, являются awaitable.

Объекты Future: Они представляют собой результат асинхронной операции, которая ещё не завершена.

Объекты Task: Являются специальными видами Future, которые оборачивают корутины.

Пример с awaitable объектами

In [ ]:
import asyncio

async def my_coroutine():
    await asyncio.sleep(1)
    return "some data"

async def main():
    result = await my_coroutine()
    print(result)

asyncio.run(main())

In [ ]:
Обработка ошибок в асинхронном программировании с использованием asyncio имеет свои особенности. В этом контексте важно учитывать, 
как обрабатывать исключения в корутинах и задачах. Давай разберём основные подходы к обработке ошибок в asyncio.

1. Обработка исключений в корутинах
Когда вы вызываете корутину с помощью await, вы можете использовать блоки try и except для обработки возможных исключений. 
Это позволяет ловить ошибки, возникающие во время выполнения асинхронных операций.

Пример обработки ошибок в корутине

In [ ]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

async def fetch_data():
    await asyncio.sleep(1)
    raise ValueError("some data error")

async def main():
    try:
        result = await fetch_data()
        print(result)
    except ValueError as e:
        print(f"error processing: {e}")
asyncio.run(main())

In [ ]:
2. Обработка исключений в задачах (Tasks)
При работе с asyncio.create_task() или asyncio.gather(), исключения, возникающие в корутинах, 
могут быть обработаны, когда вы ожидаете завершения задачи.

Пример обработки ошибок в задачах

In [ ]:
import asyncio

async def fetch_data():
    await asyncio.sleep(1)
    raise ValueError("error of load data")

async def main():
    task = asyncio.create_task(fatch_data())

    try:
        await task
    except ValueError as e:
        print(f"exception : {e}")

asyncio.run(main())

In [ ]:
import asyncio

async def failing_task():
    await asyncio.sleep(1)
    raise RuntimeError("Task failed!")

async def main():
    asyncio.create_task(failing_task())  # Ошибка "потеряется"
    await asyncio.sleep(2)  # Дадим задаче время выполнить код

asyncio.run(main())  # Ошибка не будет напечатана!


In [ ]:
import asyncio

async def failing_task():
    await asyncio.sleep(1)
    raise RuntimeError("Task failed!")

async def main():
    async def wrapper():
        try:
            await failing_task()
        except Exception as e:
            print(f"Caught an error in task: {e}")

    asyncio.create_task(wrapper())  # Оборачиваем в `try/except`
    await asyncio.sleep(2)  # Даем задаче время на выполнение

asyncio.run(main())


In [ ]:
import asyncio

async def failing_task():
    await asyncio.sleep(1)
    raise RuntimeError("Task failed!")

def handle_task_result(task):
    try:
        task.result()  # Получает результат или выбрасывает исключение
    except Exception as e:
        print(f"Task failed with error: {e}")

async def main():
    task = asyncio.create_task(failing_task())
    task.add_done_callback(handle_task_result)  # Добавляем обработчик
    await asyncio.sleep(2)

asyncio.run(main())


In [ ]:
3. Использование asyncio.gather() для обработки нескольких задач
Когда вы используете asyncio.gather(), вы можете указать, 
как обрабатывать исключения, возникающие в нескольких корутинах.

Пример использования asyncio.gather()

In [ ]:
import asyncio
import nest_asyncio

nest_asyncio.apply()

async def fetch_data(num):
    await asyncio.sleep(1)
    if num == 1:
        raise ValueError("error on data load 1")
    return f"data {num} loaded"

async def main():
    tasks = [fetch_data(i) for i in range(3)]

    try:
        results = await asyncio.gather(*tasks)
        print(results)
    except Exception as e:
        print(f"error {e}")

asyncio.run(main())


In [ ]:
4. Использование return_exceptions
asyncio.gather() принимает параметр return_exceptions, который позволяет 
продолжать выполнение других корутин, даже если одна из них вызывает исключение.

Пример с return_exceptions

In [ ]:
import asyncio

async def fetch_data(num):
    await asyncio.sleep(1)
    if num == 1:
        raise ValueError("error 1!")

    return f"data {num} loaded"
async def main():
    tasks = [fetch_data(i) for i in range(3)]
    results = await asyncio.gather(*tasks, return_exceptions=True)

    for result in results:
        if isinstance(result, Exception):
            print(f"excpetion: {result}")
        else:
            print(result)
asyncio.run(main())